# 08 · A entidade compradora ajuda o TF-IDF?

Spike E4-S3 do [`docs/backlog.md`](../docs/backlog.md). As opções e a forma de decidir foram escritas **antes** de correr, no [`experiencias/e4_s3.yaml`](../experiencias/e4_s3.yaml); este notebook lê esse ficheiro (com o mesmo `ler` dos outros spikes) e não muda nada depois de ver os números. Os números que saem daqui vão para a decisão 7 do [`docs/qualidade_dados.md`](../docs/qualidade_dados.md).

**A pergunta.** O texto de entrada é hoje o `objectoContrato` mais a `descContrato` (decisão 5). A `adjudicante` — quem compra — é uma coluna que já existe **antes** de o CPV ser escolhido, por isso pode entrar no texto. Junta-se ao início, no formato `entidade | texto` (a função `with_entity` do [`src/train_tfidf.py`](../src/train_tfidf.py)). A dúvida: a entidade ajuda (cada entidade compra sempre as mesmas coisas) ou só ensina o modelo a repetir os códigos genéricos que essa entidade costuma usar?


In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "src"))
sys.path.insert(0, str(Path.cwd().parent / "scripts"))
import baselines as bl
import comparison as cmp
import correr_experiencia as ce
import evaluation as ev
from cpv_data import code_level

RAIZ = Path.cwd().parent
DATA = RAIZ / "data" / "processed"
METRICS = RAIZ / "metrics"

EXP = ce.ler(RAIZ / "experiencias" / "e4_s3.yaml")
N_MIN, ALPHA, EPOCAS = EXP["n_min"], EXP["alpha"], EXP["epocas"]
TAU = EXP["tau"][0]
PASTAS = EXP["com_entidade"]  # nome da opção -> pasta das sugestões
GRUPOS = EXP["grupos"]

val = pd.read_parquet(DATA / "validacao.parquet",
                      columns=["idcontrato", "codigos", "adjudicante", "visto_no_treino"])
treino = pd.read_parquet(DATA / "treino.parquet", columns=["codigo", "adjudicante"])
print(f"YAML: N = {N_MIN}, alpha = {ALPHA:g}, {EPOCAS} épocas, tau = {TAU:g}")
print("opções:", PASTAS)
print(f"validação: {len(val)} contratos; treino: {len(treino)}")


YAML: N = 10, alpha = 1e-07, 24 épocas, tau = 0.25
opções: {'sem_entidade': 'metrics/e4_s1/n_min_10_ep_24', 'com_entidade': 'metrics/e4_s3/com_entidade'}
validação: 23945 contratos; treino: 478384


## Os dois treinos

O "sem entidade" é o modelo já treinado no E4-S1, com N = 10 e 24 épocas; **não se volta a treinar**, para a entidade ser a única diferença. O "com entidade" é o treino novo deste spike.


In [2]:
def nome_tau(tau):
    return "sugestoes_validacao.parquet" if tau == 0 else f"sugestoes_validacao_tau{tau:g}.parquet"


def resumo(pasta):
    return json.loads((RAIZ / pasta / "treino.json").read_text(encoding="utf-8"))


info = {nome: resumo(pasta) for nome, pasta in PASTAS.items()}
for nome, i in info.items():
    if (i["n_min"], i["alpha"], i["epocas"]) != (N_MIN, ALPHA, EPOCAS):
        raise ValueError(f"{nome}: o treino.json não é o que o YAML pede: N={i['n_min']} "
                         f"alpha={i['alpha']:g} epocas={i['epocas']}")
    if i["validacao"]["contratos"] != len(val):
        raise ValueError(f"{nome}: sugeriu para outra validação ({i['validacao']['contratos']})")
if info["com_entidade"].get("com_entidade") is not True:
    raise ValueError("o treino com entidade não diz com_entidade: true")
if info["sem_entidade"].get("com_entidade"):
    raise ValueError("o treino sem entidade diz que tem entidade")
pd.DataFrame({nome: {"N": i["n_min"], "alpha": f"{i['alpha']:g}", "épocas": i["epocas"],
                     "τ nos ficheiros": list(i["ficheiros"].values()),
                     "com entidade": bool(i.get("com_entidade")),
                     "contratos na validação": i["validacao"]["contratos"]}
              for nome, i in info.items()})


,sem_entidade,com_entidade
N,10,10
alpha,1e-07,1e-07
épocas,24,24
τ nos ficheiros,"[0.0, 0.25, 0.5]",[0.25]
com entidade,False,True
contratos na validação,23945,23945


## As sugestões

Um ficheiro por opção, o do τ escolhido no E4-S4 (0,25). As verificações são as dos notebooks 04 a 07: uma linha por contrato da validação, alinhada pela mesma ordem, com 5 códigos distintos de 8 dígitos.


In [3]:
import re

CODIGO = re.compile(r"\d{8}")


def ler_sugestoes(pasta):
    sug = pd.read_parquet(RAIZ / pasta / nome_tau(TAU), columns=["idcontrato", "sugestoes"])
    sug["idcontrato"] = sug["idcontrato"].astype(str)
    if sug["idcontrato"].duplicated().any() or set(sug["idcontrato"]) != set(val["idcontrato"].astype(str)):
        raise ValueError(f"{pasta}: os contratos não são os da validação, um por linha")
    s = sug.set_index("idcontrato")["sugestoes"].reindex(val["idcontrato"].astype(str)).map(list)
    s.index = val.index
    if not s.map(lambda l: len(set(l)) == ev.TOP_K and all(CODIGO.fullmatch(c) for c in l)).all():
        raise ValueError(f"{pasta}: sugestões mal formadas")
    return s


por = {nome: ler_sugestoes(pasta) for nome, pasta in PASTAS.items()}
pd.Series({nome: s.iloc[0] for nome, s in por.items()})


sem_entidade    [45233200, 98300000, 45233220, 45233252, 45233...
com_entidade    [45233200, 45233220, 45233252, 45233222, 45233...
dtype: object

## Os três grupos

A entidade de um contrato da validação "está no treino" se a sua `adjudicante` limpa (sem espaços a mais, como no `with_entity`) aparecer nas `adjudicante` do treino. Os grupos são os do YAML.


In [4]:
def limpar(serie):
    return serie.astype("object").fillna("").str.split().str.join(" ")


entidades_treino = set(limpar(treino["adjudicante"]))
val_entidade = limpar(val["adjudicante"])
mascara = {
    "todos": pd.Series(True, index=val.index),
    "entidades_no_treino": val_entidade.isin(entidades_treino),
    "entidades_novas": ~val_entidade.isin(entidades_treino),
}
assert list(mascara) == GRUPOS, "os grupos do YAML e do notebook têm de ser os mesmos"
pd.Series({g: int(m.sum()) for g, m in mascara.items()}, name="contratos")


todos                  23945
entidades_no_treino    23438
entidades_novas          507
Name: contratos, dtype: int64

## Os números nos três grupos

Além do top-5 no código completo e na classe, mostra-se a **parte das primeiras sugestões ao nível do grupo ou acima** (nível ≤ 3, ou seja, códigos genéricos como `45200000`). É a métrica que apanha a suspeita: se a entidade fizer o modelo cair em códigos genéricos, esta parte sobe.


In [5]:
def nivel_da_primeira(sugestoes):
    return code_level(sugestoes.map(lambda s: s[0]))


def linha(mask):
    codigos = val["codigos"][mask]
    out = {}
    for nome, s in por.items():
        rep = ev.report(s[mask], codigos, val["visto_no_treino"][mask])
        out[nome] = {
            "contratos": int(mask.sum()),
            "top-5 código": rep.loc[("todos", "codigo"), "top5"],
            "top-5 classe": rep.loc[("todos", "classe"), "top5"],
            "top-1 código": rep.loc[("todos", "codigo"), "top1"],
            "1.ª sugestão ≤ grupo": (nivel_da_primeira(s[mask]) <= 3).mean(),
        }
    return pd.DataFrame(out).T


quadro = pd.concat({g: linha(m) for g, m in mascara.items()})
quadro


contratos  top-5 código  top-5 classe  \
todos               sem_entidade    23945.0      0.783482      0.841105   
                    com_entidade    23945.0      0.807596      0.854844   
entidades_no_treino sem_entidade    23438.0      0.784270      0.840747   
                    com_entidade    23438.0      0.808200      0.854372   
entidades_novas     sem_entidade      507.0      0.714286      0.869318   
                    com_entidade      507.0      0.754464      0.892045   

                                  top-1 código  1.ª sugestão ≤ grupo  
todos               sem_entidade      0.576560              0.305742  
                    com_entidade      0.625088              0.331259  
entidades_no_treino sem_entidade      0.578905              0.307193  
                    com_entidade      0.627578              0.333518  
entidades_novas     sem_entidade      0.370536              0.238659  
                    com_entidade      0.406250              0.226824

## A decisão

A regra foi escrita no YAML antes de correr:

1. a regra de comparação do E4-T1 em **todos** os contratos, com **sem entidade** por defeito (é mais simples);
2. a entidade só entra se, além de ganhar no passo 1, as duas guardas se cumprirem:
   - não piorar o top-5 no código completo nos contratos de **entidades novas**;
   - não aumentar a parte das primeiras sugestões ao nível do grupo ou acima (nível ≤ 3) em **todos**.

Em ambas, uma diferença só conta se o bootstrap emparelhado (95%) o mostrar.


In [6]:
def piora(r, maior_e_pior):
    """A opção com entidade piora esta métrica, com uma diferença que conta?

    Em todas as comparações, `a` é o "com entidade" e `b` o "sem entidade". `maior_e_pior` diz o
    sentido da métrica: no top-5 (um acerto) maior é melhor; na parte das sugestões genéricas
    (nível <= 3), maior é pior. Sem isto, a guarda 2 lê-se ao contrário.
    """
    venceu = "a" if r["a_melhor"] > r["b_melhor"] else "b"
    return bool(r["conta"] and venceu == ("a" if maior_e_pior else "b"))


# Passo 1: a regra do E4-T1 em todos os contratos, com "sem entidade" por defeito.
duelo = cmp.compare(por["com_entidade"], por["sem_entidade"], val["codigos"])
passo1 = "com entidade" if cmp.choose_from(duelo, default="b") == "a" else "sem entidade"

# Guarda 1: top-5 no código completo nas entidades novas (acerto: maior é melhor)
novas = mascara["entidades_novas"]
guarda1 = cmp.paired_bootstrap(
    cmp.weighted_mean(ev.hits(por["com_entidade"][novas], val["codigos"][novas], "codigo", ev.TOP_K)),
    cmp.weighted_mean(ev.hits(por["sem_entidade"][novas], val["codigos"][novas], "codigo", ev.TOP_K)),
    int(novas.sum()))

# Guarda 2: parte das primeiras sugestões ao nível do grupo ou acima, em todos (genérico: maior é pior)
indicador = {nome: (nivel_da_primeira(s) <= 3).astype(float) for nome, s in por.items()}
guarda2 = cmp.paired_bootstrap(cmp.weighted_mean(indicador["com_entidade"]),
                               cmp.weighted_mean(indicador["sem_entidade"]), len(val))

p1, p2 = piora(guarda1, maior_e_pior=False), piora(guarda2, maior_e_pior=True)
entra = passo1 == "com entidade" and not p1 and not p2

print("Passo 1 (todos os contratos): fica", passo1)
print(f"  top-5 código: com entidade {duelo.loc[('codigo', 'top5'), 'a']:.2%} vs "
      f"sem entidade {duelo.loc[('codigo', 'top5'), 'b']:.2%} "
      f"(conta: {bool(duelo.loc[('codigo', 'top5'), 'conta'])})")
print(f"  top-5 classe: com entidade {duelo.loc[('classe', 'top5'), 'a']:.2%} vs "
      f"sem entidade {duelo.loc[('classe', 'top5'), 'b']:.2%} "
      f"(conta: {bool(duelo.loc[('classe', 'top5'), 'conta'])})")
print("\nGuarda 1 (top-5 código, entidades novas; maior é melhor): "
      + ("a entidade PIORA, e conta" if p1 else "a entidade não piora com diferença que conte"))
print(f"  com entidade {guarda1['a']:.2%} (vence em {guarda1['a_melhor']:.0%} das amostras) vs "
      f"sem entidade {guarda1['b']:.2%} (vence em {guarda1['b_melhor']:.0%})")
print("\nGuarda 2 (1.ª sugestão <= grupo, todos; maior é pior): "
      + ("a entidade AUMENTA, e conta" if p2 else "a entidade não aumenta com diferença que conte"))
print(f"  com entidade {guarda2['a']:.2%} (vence em {guarda2['a_melhor']:.0%} das amostras) vs "
      f"sem entidade {guarda2['b']:.2%} (vence em {guarda2['b_melhor']:.0%})")

print("\nDECISÃO (regra do YAML): a entidade "
      + ("ENTRA no texto de entrada." if entra else "NÃO entra: fica o texto da decisão 5."))


Passo 1 (todos os contratos): fica com entidade
  top-5 código: com entidade 80.76% vs sem entidade 78.35% (conta: True)
  top-5 classe: com entidade 85.48% vs sem entidade 84.11% (conta: True)

Guarda 1 (top-5 código, entidades novas; maior é melhor): a entidade não piora com diferença que conte
  com entidade 75.45% (vence em 98% das amostras) vs sem entidade 71.43% (vence em 1%)

Guarda 2 (1.ª sugestão <= grupo, todos; maior é pior): a entidade AUMENTA, e conta
  com entidade 33.13% (vence em 100% das amostras) vs sem entidade 30.57% (vence em 0%)

DECISÃO (regra do YAML): a entidade NÃO entra: fica o texto da decisão 5.


## O escolhido contra o sem entidade

A avaliação completa (as quatro medidas, todos os contratos e o texto novo) das duas opções, para se ver onde diferem.


In [7]:
lado = pd.concat({"com entidade": ev.report(por["com_entidade"], val["codigos"], val["visto_no_treino"]),
                  "sem entidade": ev.report(por["sem_entidade"], val["codigos"], val["visto_no_treino"])}, axis=1)
lado


com entidade                               sem entidade  \
                         contratos      top1      top5  f1_macro    contratos   
grupo      medida                                                               
todos      codigo            19906  0.625088  0.807596  0.285746        19906   
           classe            14047  0.659358  0.854844  0.441640        14047   
           divisao           23945  0.819002  0.939110  0.654360        23945   
           so_divisao         4039  0.695717  0.876207  0.399968         4039   
texto novo codigo            16565  0.575128  0.779294  0.273303        16565   
           classe            11959  0.620370  0.837361  0.424456        11959   
           divisao           20239  0.792085  0.929888  0.626693        20239   
           so_divisao         3674  0.672564  0.865269  0.395575         3674   

                                                     
                           top1      top5  f1_macro  
grupo      medida                                    
todos      codigo      0.576560  0.783482  0.256768  
           classe      0.624617  0.841105  0.420215  
           divisao     0.811359  0.936646  0.645037  
           so_divisao  0.692003  0.883635  0.380257  
texto novo codigo      0.520616  0.751404  0.242790  
           classe      0.582741  0.821808  0.399050  
           divisao     0.783438  0.926874  0.616768  
           so_divisao  0.668209  0.873435  0.378499

## A régua, para contexto

A régua (E4-T2) sugere sempre os códigos de treino mais frequentes. Não decide nada aqui: é uma estimativa pontual (sem bootstrap), só para confirmar que a opção fica muito acima dela.


In [8]:
regua = bl.suggest_most_common(bl.most_common_labels(treino["codigo"], n_min=N_MIN), val.index)
linhas = {}
for grupo, mask in mascara.items():
    for medida in ("codigo", "classe"):
        e = ev.report(por["sem_entidade"][mask], val["codigos"][mask], val["visto_no_treino"][mask]).loc[("todos", medida), "top5"]
        r = ev.report(regua[mask], val["codigos"][mask], val["visto_no_treino"][mask]).loc[("todos", medida), "top5"]
        linhas[(grupo, medida)] = {"sem entidade": e, "régua": r}
pd.DataFrame(linhas).T


sem entidade     régua
todos               codigo      0.783482  0.263237
                    classe      0.841105  0.155976
entidades_no_treino codigo      0.784270  0.266081
                    classe      0.840747  0.157883
entidades_novas     codigo      0.714286  0.013393
                    classe      0.869318  0.005682

## Conclusão

A entidade **não entra** no texto. A regra escrita no YAML antes de correr só a deixava entrar se não aumentasse a parte das primeiras sugestões ao nível do grupo ou acima — e aumenta.

Números na validação (outubro de 2025, 23 945 contratos, N = 10, 24 épocas, τ = 0,25):

| Grupo | top-5 código (sem → com) | top-5 classe (sem → com) | 1.ª sugestão ≤ grupo (sem → com) |
|---|---|---|---|
| todos | 78,35% → **80,76%** | 84,11% → **85,48%** | 30,57% → **33,13%** |
| entidades no treino | 78,43% → **80,82%** | 84,07% → **85,44%** | 30,72% → **33,35%** |
| entidades novas | 71,43% → **75,45%** | 86,93% → **89,20%** | 23,87% → **22,68%** |

- **Passo 1 (a regra do E4-T1, em todos):** ganha o "com entidade" — top-5 código +2,41 pontos, top-5 classe +1,37; as duas diferenças contam no bootstrap.
- **Guarda 1 (top-5 código nas entidades novas):** cumpre-se, e até a favor da entidade (+4,02 pontos; venceu em 98% das amostras do bootstrap). O receio de o modelo decorar as entidades novas não se confirmou.
- **Guarda 2 (1.ªs sugestões ao nível do grupo ou acima, em todos):** **falha.** A entidade sobe esta parte de 30,57% para 33,13% (+2,55 pontos) e a subida conta: em 1000 de 1000 amostras do bootstrap a opção com entidade tem a parte maior.

**A decisão 7 fica fechada:** o texto de entrada continua o da decisão 5 (`objectoContrato` + `descContrato` + `DescrAcordoQuadro`), sem a `adjudicante`. O modelo com entidade acerta mais códigos completos, mas apoia-se mais em códigos genéricos — exatamente o que a regra proibia, e escrita antes de ver os números. Como o modelo em vigor não muda, a régua (E4-T2) e os resultados do E4-H1 **não** precisam de ser recalculados.
